In [1]:
from google.colab import drive
import os
import pandas as pd
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_DIR = "/content/drive/MyDrive/MMRetrieval/flickr8k"
IMAGE_DIR = f"{DATASET_DIR}/Images"
CAPTION_FILE = f"{DATASET_DIR}/captions.txt"

In [3]:
print("Images:", len(os.listdir(IMAGE_DIR)))

df = pd.read_csv(CAPTION_FILE)

print(df.head())
print(df.columns)
print(df.shape)

Images: 8091
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
Index(['image', 'caption'], dtype='object')
(40455, 2)


In [4]:
images = df["image"].unique().tolist()

print("Unique images:", len(images))

Unique images: 8091


In [5]:
# Create fixed split, 80, 10, 10
train_imgs, temp_imgs = train_test_split(
    images,
    train_size=6000,
    random_state=42,
    shuffle=True
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=1000,
    random_state=42,
    shuffle=True
)

print(len(train_imgs))
print(len(val_imgs))
print(len(test_imgs))

6000
1091
1000


In [6]:
# Train test and val dataset
train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)

val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)

test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

In [7]:
# Save files in drive
import pickle

split_dict = {
    "train": train_imgs,
    "val": val_imgs,
    "test": test_imgs
}

with open(
    f"{DATASET_DIR}/flickr8k_split.pkl",
    "wb"
) as f:
    pickle.dump(split_dict, f)

In [8]:
# Test Load
with open(
    f"{DATASET_DIR}/flickr8k_split.pkl",
    "rb"
) as f:
    split_dict = pickle.load(f)

In [9]:
train_df.head()

,image,caption
0,1001773457_577c3a7d70.jpg,A black dog and a spotted dog are fighting
1,1001773457_577c3a7d70.jpg,A black dog and a tri-colored dog playing with...
2,1001773457_577c3a7d70.jpg,A black dog and a white dog with brown spots a...
3,1001773457_577c3a7d70.jpg,Two dogs of different breeds looking at each o...
4,1001773457_577c3a7d70.jpg,Two dogs on pavement moving toward each other .


Data Preprocessing

In [10]:
import re
from collections import Counter
import pickle


In [11]:
# Clean captions

def clean_caption(text):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [12]:
# Apply on the datasets

train_df["caption"] = train_df["caption"].apply(clean_caption)
val_df["caption"] = val_df["caption"].apply(clean_caption)
test_df["caption"] = test_df["caption"].apply(clean_caption)

In [13]:
# Add the specialised tokens

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

In [14]:
# Train data vocab

counter = Counter()

for caption in train_df["caption"]:
    counter.update(caption.split())

In [15]:
print("Unique words:", len(counter))

Unique words: 7744


In [16]:
# Create Vocab

MIN_FREQ = 5

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
    SOS_TOKEN: 2,
    EOS_TOKEN: 3
}

for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

idx2word = {idx: word for word, idx in vocab.items()}

In [17]:
print("Vocabulary size:", len(vocab))

Vocabulary size: 2561


In [18]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [19]:
# Set Max caption length for batching

lengths = []

for caption in train_df["caption"]:
    lengths.append(
        len(caption.split()) + 2
    )

print("Max length:", max(lengths))
print("Average length:", sum(lengths)/len(lengths))

Max length: 38
Average length: 12.790533333333334


In [20]:
MAX_LEN = 38

In [21]:
# Save and reload vocab

with open(
    f"{DATASET_DIR}/vocab.pkl",
    "wb"
) as f:
    pickle.dump(vocab, f)

In [22]:
with open(
    f"{DATASET_DIR}/vocab.pkl",
    "rb"
) as f:
    vocab = pickle.load(f)

Dataset Classes

In [23]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [24]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [25]:
from collections import defaultdict
import random

train_caption_map = defaultdict(list)
val_caption_map = defaultdict(list)
test_caption_map = defaultdict(list)

bad_img = "861608773_bdafd5c996.jpg"

train_caption_map.pop(bad_img, None)
val_caption_map.pop(bad_img, None)
test_caption_map.pop(bad_img, None)

for _, row in train_df.iterrows():
    train_caption_map[row["image"]].append(row["caption"])

for _, row in val_df.iterrows():
    val_caption_map[row["image"]].append(row["caption"])

for _, row in test_df.iterrows():
    test_caption_map[row["image"]].append(row["caption"])


class Flickr8KRetrievalDataset(Dataset):
    def __init__(self, caption_map, image_dir, vocab, transform=None, random_caption=True):
        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]      # fixed caption for eval

        try:
          image = Image.open(
              os.path.join(self.image_dir, image_name)
          ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, caption

In [26]:
class Flickr8KAllCaptionEvalDataset(Dataset):

    def __init__(self, caption_map, image_dir, vocab, transform=None):

        self.samples = []
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform

        for image_name in sorted(caption_map.keys()):

            for caption in caption_map[image_name]:

                self.samples.append(
                    (image_name, caption)
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_name, caption = self.samples[idx]

        image = Image.open(
            os.path.join(self.image_dir, image_name)
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        caption = torch.tensor(
            encode_caption(caption, self.vocab),
            dtype=torch.long
        )

        return image, torch.tensor(caption)

In [27]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    captions = []
    lengths = []

    for image, caption in batch:
        images.append(image)
        captions.append(caption)
        lengths.append(len(caption))

    images = torch.stack(images)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    lengths = torch.tensor(lengths)

    return images, captions, lengths

In [28]:
train_dataset = Flickr8KRetrievalDataset(
    train_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=True
)

val_dataset = Flickr8KRetrievalDataset(
    val_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=False
)

test_dataset = Flickr8KRetrievalDataset(
    test_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform,
    random_caption=False
)

In [29]:
all_caption_test_dataset = Flickr8KAllCaptionEvalDataset(
    test_caption_map,
    IMAGE_DIR,
    vocab,
    image_transform
)

In [30]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

6000
1091
1000


In [31]:
# Dataloaders for clip style loading
BATCH_SIZE = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [32]:
all_caption_test_loader = DataLoader(
    all_caption_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [33]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [34]:
device

'cuda'

In [35]:
print("device" in globals())
print("text_encoder" in globals())
print("train_loader" in globals())

True
False
True


In [36]:
image, caption = train_dataset[0]

print(image.shape)

print(caption)

torch.Size([3, 224, 224])
tensor([ 2, 26, 27, 16, 32, 33, 34, 14, 15,  3])


ResNet50 Image Encoder

In [37]:
EMBED_DIM = 256

In [38]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

In [39]:
# Encoder Image
class ImageEncoder(nn.Module):

    def __init__(self, embed_dim=768):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        for param in backbone.parameters():
          param.requires_grad = False

        # Fine tune last two ResNet stages
        for param in backbone.layer3.parameters():
            param.requires_grad = True

        for param in backbone.layer4.parameters():
            param.requires_grad = True

        self.backbone = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, images):

        features = self.backbone(images)

        features = features.squeeze(-1).squeeze(-1)

        embeddings = self.projection(features)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [42]:
#Text Encoder
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)

class TextEncoderAttention(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=300,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            dropout=0.5,
            bidirectional=True,
            batch_first=True
        )

        # Attention layer
        self.attn_W = nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_U = nn.Linear(hidden_dim * 2, hidden_dim)
        self.attn_v = nn.Linear(hidden_dim, 1, bias=False)

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 768)
        )

    def forward(
        self,
        captions,
        lengths
    ):

        embedded = self.embedding(captions)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (hidden, cell) = self.lstm(packed)

        outputs, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        # outputs:
        # (batch, seq_len, hidden_dim*2)
        batch_size = outputs.size(0)
        seq_len = outputs.size(1)

        # Final BiLSTM hidden state (query)
        hidden_forward = hidden[-2]
        hidden_backward = hidden[-1]

        query = torch.cat(
            [hidden_forward, hidden_backward],
            dim=1
        )

        query = query.unsqueeze(1).repeat(
            1,
            seq_len,
            1
        )

        # Bahdanau Attention
        energy = torch.tanh(
            self.attn_W(outputs)
            +
            self.attn_U(query)
        )

        attn_scores = self.attn_v(energy).squeeze(-1)

        mask = (
            torch.arange(
                seq_len,
                device=outputs.device
            )
            .expand(batch_size, seq_len)
            >= lengths.unsqueeze(1).to(outputs.device)
        )

        attn_scores = attn_scores.masked_fill(
            mask,
            -1e9
        )

        attn_weights = torch.softmax(
            attn_scores,
            dim=1
        )

        context = torch.sum(
            outputs *
            attn_weights.unsqueeze(-1),
            dim=1
        )



        embeddings = self.projection(context)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [43]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


In [44]:
encoder = ImageEncoder(
    embed_dim=768
).to(device)

images, captions, lengths = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    image_embeddings = encoder(images)

print(image_embeddings.shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 236MB/s]


torch.Size([256, 768])


In [45]:
# text encoder installation and test
text_encoder = TextEncoderAttention(
    vocab_size=len(vocab),
    embed_dim=300,
    hidden_dim=512,
    pad_idx=vocab["<PAD>"]
).to(device)

images, captions, lengths = next(iter(train_loader))

captions = captions.to(device)

with torch.no_grad():
    text_embeddings = text_encoder(
        captions,
        lengths
    )

print(text_embeddings.shape)

torch.Size([256, 768])


In [46]:
# compatibility verification
with torch.no_grad():
    image_embeddings = encoder(images.to(device))

    text_embeddings = text_encoder(
        captions.to(device),
        lengths
    )

print(image_embeddings.shape)
print(text_embeddings.shape)

torch.Size([256, 768])
torch.Size([256, 768])


In [47]:
# Normalize embedding
import torch.nn.functional as F

image_embeddings = F.normalize(image_embeddings, dim=1)
text_embeddings = F.normalize(text_embeddings, dim=1)

print(image_embeddings.norm(dim=1).mean())
print(text_embeddings.norm(dim=1).mean())

tensor(1., device='cuda:0')
tensor(1., device='cuda:0')


In [48]:
# Joint Model
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ResNetLSTMRetrieval(nn.Module):
    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable temperature parameter
        self.logit_scale = nn.Parameter(
            torch.ones([]) * np.log(1 / 0.07)
        )

    def forward(self, images, captions, lengths):
        image_emb = self.image_encoder(images)
        text_emb = self.text_encoder(captions, lengths)

        # Normalize embeddings
        image_emb = F.normalize(image_emb, p=2, dim=1)
        text_emb = F.normalize(text_emb, p=2, dim=1)

        return image_emb, text_emb



model = ResNetLSTMRetrieval(
    image_encoder=encoder,
    text_encoder=text_encoder
).to(device)

for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [49]:
# CLIP-Style Contrastive Loss
def clip_contrastive_loss(image_emb, text_emb, logit_scale):
    # Similarity matrix: [B, B]
    logits = torch.matmul(image_emb, text_emb.T)
    logits = logits * logit_scale.exp()

    targets = torch.arange(
        image_emb.size(0),
        device=image_emb.device
    )

    loss_i2t = F.cross_entropy(logits, targets, label_smoothing=0.1)
    loss_t2i = F.cross_entropy(logits.T, targets, label_smoothing=0.1)

    loss = (loss_i2t + loss_t2i) / 2

    return loss, logits

In [50]:
# Forward pass test
images, captions, lengths = next(iter(train_loader))

images = images.to(device)
captions = captions.to(device)

with torch.no_grad():
    image_emb, text_emb = model(
        images,
        captions,
        lengths
    )

print(image_emb.shape)
print(text_emb.shape)

torch.Size([256, 768])
torch.Size([256, 768])


In [51]:
# Loss test
loss, logits = clip_contrastive_loss(
    image_emb,
    text_emb,
    model.logit_scale
)

print(loss.item())
print(logits.shape)

5.6435089111328125
torch.Size([256, 256])


In [52]:

# Training
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.image_encoder.backbone.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.image_encoder.projection.parameters(),
            "lr": 3e-4
        },
        {
            "params": model.text_encoder.parameters(),
            "lr": 3e-4
        },
        {
            "params": [model.logit_scale],
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)

In [53]:
# Add Validation Loop
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            image_emb, text_emb = model(
                images,
                captions,
                lengths
            )

            loss, _ = clip_contrastive_loss(
                image_emb,
                text_emb,
                model.logit_scale
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [54]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
del images, captions
gc.collect()
torch.cuda.empty_cache()

In [56]:
# training
NUM_EPOCHS = 10
best_val_loss = float("inf")
BEST_MODEL_PATH = "/content/drive/MyDrive/MMRetrieval/resnet_lstm_m1/best_retrieval_model_attention.pth"

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, captions, lengths) in enumerate(train_loader):
        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        image_emb, text_emb = model(images, captions, lengths)
        loss, _ = clip_contrastive_loss(
            image_emb,
            text_emb,
            model.logit_scale
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )
        optimizer.step()
        with torch.no_grad():
            model.logit_scale.clamp_(0, np.log(100))

        running_loss += loss.item()

        if batch_idx % 20 == 0:
          print(
              f"Epoch {epoch+1} | "
              f"Batch {batch_idx}/{len(train_loader)} | "
              f"Loss: {loss.item():.4f}"
          )

    train_loss = running_loss / len(train_loader)
    val_loss = evaluate(model, val_loader, device)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f}"
    )

    scheduler.step()

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss
            },
            BEST_MODEL_PATH
        )
        print("  ✓ Saved best model")

Epoch 1 | Batch 0/24 | Loss: 5.1234
Epoch 1 | Batch 20/24 | Loss: 4.5892
Epoch 01 | Train: 4.8083 | Val: 4.3111
  ✓ Saved best model
Epoch 2 | Batch 0/24 | Loss: 4.4491
Epoch 2 | Batch 20/24 | Loss: 4.1700
Epoch 02 | Train: 4.2617 | Val: 3.9988
  ✓ Saved best model
Epoch 3 | Batch 0/24 | Loss: 4.0140
Epoch 3 | Batch 20/24 | Loss: 3.7637
Epoch 03 | Train: 3.8962 | Val: 3.7942
  ✓ Saved best model
Epoch 4 | Batch 0/24 | Loss: 3.7531
Epoch 4 | Batch 20/24 | Loss: 3.5479
Epoch 04 | Train: 3.5947 | Val: 3.6590
  ✓ Saved best model
Epoch 5 | Batch 0/24 | Loss: 3.4677
Epoch 5 | Batch 20/24 | Loss: 3.4448
Epoch 05 | Train: 3.3541 | Val: 3.5520
  ✓ Saved best model
Epoch 6 | Batch 0/24 | Loss: 3.2120
Epoch 6 | Batch 20/24 | Loss: 3.0801
Epoch 06 | Train: 3.1504 | Val: 3.4595
  ✓ Saved best model
Epoch 7 | Batch 0/24 | Loss: 3.0085
Epoch 7 | Batch 20/24 | Loss: 3.1788
Epoch 07 | Train: 3.0127 | Val: 3.3954
  ✓ Saved best model
Epoch 8 | Batch 0/24 | Loss: 2.9532
Epoch 8 | Batch 20/24 | Loss: 2.8

In [57]:
FINAL_MODEL_PATH = "/content/drive/MyDrive/MMRetrieval/resnet_lstm_m1/best_retrieval_model_attention.pth"

torch.save(model.state_dict(), BEST_MODEL_PATH)

Evaluation

In [58]:
def extract_embeddings(model, dataloader, device):
    model.load_state_dict(
    torch.load(FINAL_MODEL_PATH, map_location=device)
    )

    model.eval()


    image_embeddings = []
    text_embeddings = []

    with torch.no_grad():
        for images, captions, lengths in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            img_emb, txt_emb = model(images, captions, lengths)

            image_embeddings.append(img_emb.cpu())
            text_embeddings.append(txt_emb.cpu())

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    return image_embeddings, text_embeddings

In [59]:
image_embs, text_embs = extract_embeddings(
    model,
    all_caption_test_loader,
    device
)

print(image_embs.shape)
print(text_embs.shape)

/tmp/ipykernel_554/340369310.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return image, torch.tensor(caption)


torch.Size([5000, 768])
torch.Size([5000, 768])


In [60]:
# Similarity Matrix
unique_image_embs = image_embs[::5]

similarity = unique_image_embs @ text_embs.T

print("Image embeddings:", unique_image_embs.shape)
print("Text embeddings:", text_embs.shape)
print("Similarity:", similarity.shape)

Image embeddings: torch.Size([1000, 768])
Text embeddings: torch.Size([5000, 768])
Similarity: torch.Size([1000, 5000])


In [61]:
# image -> text
def image_to_text_recall(similarity, k):

    correct = 0

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        topk = similarity[img_idx].topk(k).indices.tolist()

        if any(idx in gt_caps for idx in topk):
            correct += 1

    return correct / similarity.shape[0]


In [62]:
def text_to_image_recall(similarity, k):

    similarity_t = similarity.T

    correct = 0

    for cap_idx in range(similarity_t.shape[0]):

        gt_img = cap_idx // 5

        topk = similarity_t[cap_idx]\
            .topk(k)\
            .indices\
            .tolist()

        if gt_img in topk:
            correct += 1

    return correct / similarity_t.shape[0]


In [63]:
def image_to_text_mrr(similarity):

    reciprocal_ranks = []

    for img_idx in range(similarity.shape[0]):

        gt_caps = set(
            range(
                img_idx * 5,
                img_idx * 5 + 5
            )
        )

        sorted_idx = torch.argsort(
            similarity[img_idx],
            descending=True
        )

        best_rank = float("inf")

        for cap in gt_caps:

            rank = (
                (sorted_idx == cap)
                .nonzero(as_tuple=True)[0]
                .item()
            ) + 1

            best_rank = min(best_rank, rank)

        reciprocal_ranks.append(
            1.0 / best_rank
        )

    return np.mean(reciprocal_ranks)

In [64]:
def text_to_image_mrr(similarity):

    similarity_t2i = similarity.T

    reciprocal_ranks = []

    for cap_idx in range(similarity_t2i.shape[0]):

        gt_image = cap_idx // 5

        sorted_idx = torch.argsort(
            similarity_t2i[cap_idx],
            descending=True
        )

        rank = (
            (sorted_idx == gt_image)
            .nonzero(as_tuple=True)[0]
            .item()
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return np.mean(reciprocal_ranks)

In [65]:
results = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@5",
        "Recall@10",
        "MRR"
    ],
    "Image→Text": [
        image_to_text_recall(similarity, 1),
        image_to_text_recall(similarity, 5),
        image_to_text_recall(similarity, 10),
        image_to_text_mrr(similarity)
    ],
    "Text→Image": [
        text_to_image_recall(similarity, 1),
        text_to_image_recall(similarity, 5),
        text_to_image_recall(similarity, 10),
        text_to_image_mrr(similarity)
    ]
})

results["Image→Text"] = results["Image→Text"].round(6)
results["Text→Image"] = results["Text→Image"].round(6)

display(results)

,Metric,Image→Text,Text→Image
0,Recall@1,0.203000,0.131800
1,Recall@5,0.446000,0.368200
2,Recall@10,0.572000,0.510400
3,MRR,0.322999,0.249877
